# 🗑️ Waste Classification — Image Preprocessing (PyTorch)

**Dataset:** `waste_dataset/` — 6 classes: glass, metal, paper, cardboard, trash, plastic  
**Format:** `.jpg` images organized in class folders  
**Framework:** PyTorch + torchvision

### Pipeline:
1. Setup & Imports
2. Explore Dataset Structure
3. Visualize Raw Samples
4. Define Transforms (Train / Val / Test)
5. Create Datasets & DataLoaders
6. Verify Preprocessing (visualize augmented samples)
7. Class Distribution & Dataset Stats
8. Check for Corrupted Images
9. Compute Mean & Std (for normalization)
10. Save Preprocessed DataLoaders Info

## 1. Setup & Imports

In [ ]:
import subprocess, sys
def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for pkg in ['torch', 'torchvision', 'matplotlib', 'seaborn', 'Pillow', 'numpy', 'tqdm']:
    install(pkg)

print('✅ All packages ready.')

In [ ]:
import os
import json
import warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from pathlib import Path
from PIL import Image, UnidentifiedImageError
from collections import Counter
from tqdm import tqdm

import torch
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split, Subset

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

# ── Device ──────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch  : {torch.__version__}')
print(f'Torchvision: {torchvision.__version__}')
print(f'Device   : {DEVICE}')

## 2. Explore Dataset Structure

In [ ]:
# ── CONFIG ───────────────────────────────────────────────────────────────
DATA_DIR    = Path('waste_dataset')   # ← change if your path is different
IMG_SIZE    = 224                     # standard for pretrained CNNs
BATCH_SIZE  = 32
NUM_WORKERS = 2
SEED        = 42
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
# TEST_RATIO  = 0.15  (remainder)

torch.manual_seed(SEED)
np.random.seed(SEED)

# ── Explore folders ──────────────────────────────────────────────────────
assert DATA_DIR.exists(), f'❌ Dataset folder not found: {DATA_DIR}'

class_folders = sorted([d for d in DATA_DIR.iterdir() if d.is_dir()])
print(f'📁 Dataset root   : {DATA_DIR.resolve()}')
print(f'🗂️  Classes found  : {len(class_folders)}')
print()

total_images = 0
class_counts = {}
for folder in class_folders:
    imgs = list(folder.glob('*.jpg')) + list(folder.glob('*.JPG'))
    class_counts[folder.name] = len(imgs)
    total_images += len(imgs)
    print(f'  {folder.name:12s} → {len(imgs):5,} images')

print(f'\n  {"TOTAL":12s} → {total_images:5,} images')

In [ ]:
# ── Sample image info ────────────────────────────────────────────────────
sample_imgs = list(DATA_DIR.rglob('*.jpg'))[:5]
print('Sample image sizes (W × H × C):')
for p in sample_imgs:
    img = Image.open(p).convert('RGB')
    arr = np.array(img)
    print(f'  {p.parent.name}/{p.name:20s}  →  {img.size[0]}×{img.size[1]}  mode={img.mode}  dtype={arr.dtype}')

## 3. Visualize Raw Samples

In [ ]:
CLASSES = sorted(class_counts.keys())
COLORS  = plt.cm.Set2(np.linspace(0, 1, len(CLASSES)))

fig, axes = plt.subplots(2, len(CLASSES)//2 + len(CLASSES)%2,
                          figsize=(18, 7))
axes = axes.flatten()

for idx, cls in enumerate(CLASSES):
    imgs = list((DATA_DIR / cls).glob('*.jpg'))
    if imgs:
        img = Image.open(imgs[0]).convert('RGB')
        axes[idx].imshow(img)
        axes[idx].set_title(cls.upper(), fontsize=12, fontweight='bold',
                             color='white',
                             bbox=dict(facecolor=COLORS[idx], alpha=0.85,
                                       boxstyle='round,pad=0.3'))
        axes[idx].axis('off')

for ax in axes[len(CLASSES):]:
    ax.set_visible(False)

plt.suptitle('Raw Sample Images — One Per Class', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 4. Define Transforms

In [ ]:
# ── ImageNet mean & std (standard for pretrained CNNs) ───────────────────
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ── TRAIN transforms — with augmentation ─────────────────────────────────
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),   # resize slightly bigger
    transforms.RandomCrop(IMG_SIZE),                      # random crop to IMG_SIZE
    transforms.RandomHorizontalFlip(p=0.5),               # flip left/right
    transforms.RandomVerticalFlip(p=0.2),                 # flip up/down
    transforms.RandomRotation(degrees=20),                # rotate ±20°
    transforms.ColorJitter(
        brightness=0.3,
        contrast=0.3,
        saturation=0.3,
        hue=0.1
    ),                                                    # color augmentation
    transforms.RandomGrayscale(p=0.05),                   # occasionally grayscale
    transforms.ToTensor(),                                # PIL → tensor [0,1]
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    ),                                                    # normalize
])

# ── VAL / TEST transforms — NO augmentation, only resize + normalize ─────
val_test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    ),
])

print('✅ Transforms defined.')
print()
print('TRAIN transforms:')
print(train_transforms)
print()
print('VAL/TEST transforms:')
print(val_test_transforms)

## 5. Create Datasets & DataLoaders

In [ ]:
# ── Load full dataset (without augmentation first, for splitting) ─────────
full_dataset = datasets.ImageFolder(
    root=str(DATA_DIR),
    transform=val_test_transforms   # neutral transform for split
)

CLASS_NAMES = full_dataset.classes
NUM_CLASSES = len(CLASS_NAMES)
print(f'Classes  : {CLASS_NAMES}')
print(f'Num classes: {NUM_CLASSES}')
print(f'Total images: {len(full_dataset):,}')
print(f'Class → Index: {full_dataset.class_to_idx}')

In [ ]:
# ── Stratified split ─────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split as sk_split

all_indices = list(range(len(full_dataset)))
all_labels  = [full_dataset.targets[i] for i in all_indices]

# Train vs (Val+Test)
train_idx, temp_idx = sk_split(
    all_indices,
    test_size=(1 - TRAIN_RATIO),
    stratify=all_labels,
    random_state=SEED
)

# Val vs Test
temp_labels = [all_labels[i] for i in temp_idx]
val_idx, test_idx = sk_split(
    temp_idx,
    test_size=0.50,
    stratify=temp_labels,
    random_state=SEED
)

print(f'Train : {len(train_idx):>5,} ({len(train_idx)/len(full_dataset)*100:.1f}%)')
print(f'Val   : {len(val_idx):>5,} ({len(val_idx)/len(full_dataset)*100:.1f}%)')
print(f'Test  : {len(test_idx):>5,} ({len(test_idx)/len(full_dataset)*100:.1f}%)')

In [ ]:
# ── Build datasets with correct transforms ────────────────────────────────

# Train dataset with augmentation
train_dataset_aug = datasets.ImageFolder(
    root=str(DATA_DIR),
    transform=train_transforms
)

train_dataset = Subset(train_dataset_aug, train_idx)
val_dataset   = Subset(full_dataset,      val_idx)
test_dataset  = Subset(full_dataset,      test_idx)

# ── DataLoaders ───────────────────────────────────────────────────────────
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == 'cuda')
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == 'cuda')
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == 'cuda')
)

print(f'Train batches : {len(train_loader)}')
print(f'Val batches   : {len(val_loader)}')
print(f'Test batches  : {len(test_loader)}')

# Quick shape check
imgs, labels = next(iter(train_loader))
print(f'\nBatch shape   : {imgs.shape}   (B × C × H × W)')
print(f'Labels shape  : {labels.shape}')
print(f'Pixel range   : [{imgs.min():.3f}, {imgs.max():.3f}]')

## 6. Verify Preprocessing (Visualize Augmented Samples)

In [ ]:
def denormalize(tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    """Reverse ImageNet normalization for visualization."""
    t = tensor.clone()
    for c, m, s in zip(range(3), mean, std):
        t[c] = t[c] * s + m
    return t.clamp(0, 1)

def show_batch(loader, title='Batch', n=8):
    imgs, labels = next(iter(loader))
    imgs_show = [denormalize(imgs[i]) for i in range(min(n, len(imgs)))]
    grid = torchvision.utils.make_grid(imgs_show, nrow=4, padding=4)
    np_img = grid.permute(1, 2, 0).numpy()

    fig, ax = plt.subplots(figsize=(16, 5))
    ax.imshow(np_img)
    ax.axis('off')
    label_names = [CLASS_NAMES[l.item()] for l in labels[:n]]
    ax.set_title(f'{title}\nLabels: {label_names}',
                  fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()

print('=== TRAIN batch (with augmentation) ===')
show_batch(train_loader, title='Train Batch — Augmented')

print('=== VAL batch (no augmentation) ===')
show_batch(val_loader, title='Val Batch — No Augmentation')

In [ ]:
# ── Show same image with vs without augmentation ──────────────────────────
sample_path = list((DATA_DIR / CLASSES[0]).glob('*.jpg'))[0]
raw_img = Image.open(sample_path).convert('RGB')

fig, axes = plt.subplots(2, 5, figsize=(18, 7))

# Row 1: original
for ax in axes[0]:
    ax.imshow(raw_img)
    ax.set_title('Original', fontsize=9)
    ax.axis('off')

# Row 2: augmented versions
for ax in axes[1]:
    aug = train_transforms(raw_img)
    aug_vis = denormalize(aug).permute(1, 2, 0).numpy()
    ax.imshow(aug_vis)
    ax.set_title('Augmented', fontsize=9, color='steelblue')
    ax.axis('off')

plt.suptitle(f'Original vs Augmented — {CLASSES[0]}',
              fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Class Distribution & Dataset Stats

In [ ]:
def get_label_counts(subset, class_names):
    labels = [subset.dataset.targets[i] for i in subset.indices]
    counts = Counter(labels)
    return {class_names[k]: v for k, v in sorted(counts.items())}

train_counts = get_label_counts(train_dataset, CLASS_NAMES)
val_counts   = get_label_counts(val_dataset,   CLASS_NAMES)
test_counts  = get_label_counts(test_dataset,  CLASS_NAMES)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
palette = sns.color_palette('Set2', NUM_CLASSES)

for ax, counts, title in zip(axes,
                               [train_counts, val_counts, test_counts],
                               ['Train', 'Validation', 'Test']):
    bars = ax.bar(counts.keys(), counts.values(),
                   color=palette, edgecolor='white', linewidth=1.5)
    ax.set_title(f'{title} — Class Distribution', fontweight='bold')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=30)
    for bar, val in zip(bars, counts.values()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                str(val), ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

# Imbalance check
all_counts = list(class_counts.values())
imbalance_ratio = max(all_counts) / min(all_counts)
print(f'\nClass imbalance ratio (max/min): {imbalance_ratio:.2f}')
if imbalance_ratio > 3:
    print('⚠️  Significant imbalance detected — consider WeightedRandomSampler or class weights.')
else:
    print('✅ Classes are fairly balanced.')

In [ ]:
# ── Compute class weights for imbalanced datasets ─────────────────────────
from torch.utils.data import WeightedRandomSampler

label_list   = [full_dataset.targets[i] for i in train_idx]
class_freq   = Counter(label_list)
n_samples    = len(label_list)

# Weight per class = total / (n_classes × count)
class_weights = {
    cls_idx: n_samples / (NUM_CLASSES * count)
    for cls_idx, count in class_freq.items()
}

sample_weights = torch.tensor(
    [class_weights[lbl] for lbl in label_list], dtype=torch.float
)

weighted_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

# Balanced DataLoader (use instead of train_loader if classes are imbalanced)
balanced_train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=weighted_sampler,        # ← replaces shuffle=True
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == 'cuda')
)

print('Class weights (for WeightedRandomSampler):')
for idx, w in sorted(class_weights.items()):
    print(f'  {CLASS_NAMES[idx]:12s} → weight = {w:.4f}')

## 8. Check for Corrupted Images

In [ ]:
corrupted = []
all_paths = list(DATA_DIR.rglob('*.jpg')) + list(DATA_DIR.rglob('*.JPG'))

print(f'Checking {len(all_paths):,} images...')
for path in tqdm(all_paths, desc='Validating images'):
    try:
        img = Image.open(path)
        img.verify()   # checks file integrity
    except (UnidentifiedImageError, Exception):
        corrupted.append(str(path))

if corrupted:
    print(f'\n⚠️  {len(corrupted)} corrupted images found:')
    for p in corrupted:
        print(f'   {p}')
else:
    print(f'\n✅ All {len(all_paths):,} images are valid!')

## 9. Compute Dataset Mean & Std (from your actual data)

In [ ]:
# ── Compute per-channel mean & std of the TRAINING set ───────────────────
# (Use these instead of ImageNet stats for best accuracy)

no_norm_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

tmp_dataset = datasets.ImageFolder(root=str(DATA_DIR), transform=no_norm_transform)
tmp_subset  = Subset(tmp_dataset, train_idx)
tmp_loader  = DataLoader(tmp_subset, batch_size=64, shuffle=False, num_workers=NUM_WORKERS)

mean = torch.zeros(3)
std  = torch.zeros(3)
n_pixels = 0

print('Computing mean & std over training set...')
for imgs, _ in tqdm(tmp_loader, desc='Pass 1 — mean'):
    b, c, h, w = imgs.shape
    mean += imgs.mean(dim=[0, 2, 3]) * b
    n_pixels += b
mean /= n_pixels

for imgs, _ in tqdm(tmp_loader, desc='Pass 2 — std'):
    b = imgs.shape[0]
    std += ((imgs - mean.view(1, 3, 1, 1)) ** 2).mean(dim=[0, 2, 3]) * b
std = torch.sqrt(std / n_pixels)

DATASET_MEAN = mean.tolist()
DATASET_STD  = std.tolist()

print(f'\n📊 Dataset Mean : {[round(m, 4) for m in DATASET_MEAN]}')
print(f'📊 Dataset Std  : {[round(s, 4) for s in DATASET_STD]}')
print(f'\n(ImageNet Mean  : {IMAGENET_MEAN})')
print(f'(ImageNet Std   : {IMAGENET_STD})')

In [ ]:
# ── Optional: Rebuild transforms using YOUR dataset stats ─────────────────
# Uncomment if dataset mean/std differ significantly from ImageNet

# train_transforms_custom = transforms.Compose([
#     transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
#     transforms.RandomCrop(IMG_SIZE),
#     transforms.RandomHorizontalFlip(p=0.5),
#     transforms.RandomVerticalFlip(p=0.2),
#     transforms.RandomRotation(degrees=20),
#     transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=DATASET_MEAN, std=DATASET_STD),
# ])

print('✅ Custom stats computed and ready to use.')

## 10. Save Preprocessing Config

In [ ]:
os.makedirs('processed', exist_ok=True)

config = {
    'img_size'       : IMG_SIZE,
    'batch_size'     : BATCH_SIZE,
    'num_classes'    : NUM_CLASSES,
    'class_names'    : CLASS_NAMES,
    'class_to_idx'   : full_dataset.class_to_idx,
    'splits'         : {
        'train': len(train_idx),
        'val'  : len(val_idx),
        'test' : len(test_idx),
    },
    'imagenet_mean'  : IMAGENET_MEAN,
    'imagenet_std'   : IMAGENET_STD,
    'dataset_mean'   : [round(m, 4) for m in DATASET_MEAN],
    'dataset_std'    : [round(s, 4) for s in DATASET_STD],
    'class_weights'  : {CLASS_NAMES[k]: round(v, 4)
                        for k, v in class_weights.items()},
    'corrupted_images': corrupted,
    'imbalance_ratio': round(imbalance_ratio, 3),
}

with open('processed/preprocessing_config.json', 'w') as f:
    json.dump(config, f, indent=2)

# Save split indices
np.save('processed/train_indices.npy', np.array(train_idx))
np.save('processed/val_indices.npy',   np.array(val_idx))
np.save('processed/test_indices.npy',  np.array(test_idx))

print('✅ Saved to ./processed/')
print('   ├── preprocessing_config.json')
print('   ├── train_indices.npy')
print('   ├── val_indices.npy')
print('   └── test_indices.npy')
print()
print('='*55)
print('  PREPROCESSING COMPLETE ✅')
print('='*55)
print(f'  Classes     : {CLASS_NAMES}')
print(f'  Train/Val/Test: {len(train_idx)} / {len(val_idx)} / {len(test_idx)}')
print(f'  Image size  : {IMG_SIZE}×{IMG_SIZE}')
print(f'  Batch size  : {BATCH_SIZE}')
print(f'  Device      : {DEVICE}')
print('='*55)
print()
print('Next step → use train_loader, val_loader, test_loader')
print('with your model (ResNet / EfficientNet / etc.)')